In [38]:
import json
import re
import unicodedata
from pathlib import Path
from sklearn.model_selection import train_test_split

In [39]:
#pretprocesiranje

In [40]:
#Ucitavanje chunkova

In [79]:
PROJECT_ROOT = Path.cwd()

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks.jsonl"
)


In [80]:
def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                records.append(record)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}."
                ) from error

    return records

In [81]:
chunks = load_jsonl(CHUNKS_PATH)
print(f"Učitano je {len(chunks)} chunkova.")

Učitano je 344 chunkova.


In [82]:
#Sređivanje teksta
#Normalizacija unicode znakova
#Spajanje prelomljenih reči
#Uklanjanje neispravnih znakova,visestrukih redova i razmaka sa čuvanjem pasusa

In [83]:
def preprocess_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = re.sub(
        r"(?<=\w)-\s*\n\s*(?=\w)",
        "",
        text
    )
    text = text.replace("�", "")

    paragraphs = re.split(r"\n\s*\n+", text)
    cleaned_paragraphs = []

    for paragraph in paragraphs:
        paragraph = re.sub(r"\s*\n\s*", " ", paragraph)
        paragraph = re.sub(r"[ \t]+", " ", paragraph)
        paragraph = paragraph.strip()

        if paragraph:
            cleaned_paragraphs.append(paragraph)

    return "\n\n".join(cleaned_paragraphs)

In [84]:
processed_chunks = []

for chunk in chunks:
    processed_chunk = chunk.copy()
    processed_chunk["processed_text"] = preprocess_text(
        chunk.get("text", "")
    )
    processed_chunks.append(processed_chunk)

In [85]:
#trenutna provera obrisati posle
print(f"Broj originalnih chunkova: {len(chunks)}")
print(f"Broj obrađenih chunkova: {len(processed_chunks)}")

Broj originalnih chunkova: 344
Broj obrađenih chunkova: 344


In [86]:
#trenutna provera obrisati posle
example_chunk = processed_chunks[0]

print("PRE SREĐIVANJA:\n")
print(example_chunk["text"][:700])

print("\nPOSLE SREĐIVANJA:\n")
print(example_chunk["processed_text"][:700])

PRE SREĐIVANJA:

Pregled

1.1
Upravljanje kvali-
tetom softvera . .
4

▶Koji procesi su važni za kvalitet softvera?
▶Koji su standardi kvaliteta softvera najbitnĳi i šta
oni definišu?

1.2
Standardi . . . . .
4

1.3
Atributi kvaliteta
softvera . . . . . .
5

▶Kojim se atributima opisuje kvalitet softvera?
▶Koji atributi kvaliteta softvera su važni za bankar-
ske aplikacĳe, koji za autonomnu vožnju, koji za
kalkulator, koji za onlajn prodaju karata, a koji za
sisteme za razmenu poruka?

Tokom poslednjih godina, IT industrĳa se brzo razvi-
ja i predstavlja jednu od najdinamičnĳih industrĳa u
svetu. Softver se razvĳa za veoma raznovrsne uređaje i
svrhe. Ove namene obuhvataju oblasti poput interneta
stvari, virt

POSLE SREĐIVANJA:

Pregled

1.1 Upravljanje kvalitetom softvera . . 4

▶Koji procesi su važni za kvalitet softvera? ▶Koji su standardi kvaliteta softvera najbitniji i šta oni definišu?

1.2 Standardi . . . . . 4

1.3 Atributi kvaliteta softvera . . . . . . 5

▶Kojim se atributima 

In [87]:
empty_chunks = [
    chunk["chunk_id"]
    for chunk in processed_chunks
    if not chunk["processed_text"]
]

print(f"Broj praznih chunkova: {len(empty_chunks)}")

if empty_chunks:
    print("Prazni chunkovi:", empty_chunks)

Broj praznih chunkova: 0


In [88]:
#ucitavanje pitanja


In [89]:
QA_PATH = (
    PROJECT_ROOT
    / "data"
    / "questions"
    / "questions_answers.json"
)

with QA_PATH.open("r", encoding="utf-8") as file:
    qa_examples = json.load(file)

print(f"Učitano pitanja: {len(qa_examples)}")

Učitano pitanja: 143


In [90]:
processed_qa = []

for example in qa_examples:
    processed_example = example.copy()

    processed_example["processed_question"] = preprocess_text(
        example.get("question", "")
    )

    processed_example["processed_answer"] = preprocess_text(
        example.get("answer", "")
    )

    source = example.get("source", {})
    source_pages = source.get("page", [])

    if isinstance(source_pages, int):
        source_pages = [source_pages]

    processed_example["source_pages"] = source_pages

    processed_qa.append(processed_example)


In [91]:
valid_qa = []

for example in processed_qa:
    question = example.get("processed_question", "")
    answer = example.get("processed_answer", "")
    source_pages = example.get("source_pages", [])

    if question and answer and source_pages:
        valid_qa.append(example)

print(f"Broj pitanja spremnih za podelu: {len(valid_qa)}")

Broj pitanja spremnih za podelu: 143


In [92]:
#Podela na trening test i validacioni skup

In [93]:
train_data, temporary_data = train_test_split(
    valid_qa,
    test_size=0.30,
    random_state=42,
    shuffle=True,
    stratify=[
        example["topic"]
        for example in valid_qa
    ]
)

In [94]:
validation_data, test_data = train_test_split(
    temporary_data,
    test_size=0.50,
    random_state=42,
    shuffle=True,
    stratify=[
        example["topic"]
        for example in temporary_data
    ]
)

In [95]:
#Moja provera obrisati pre predaje da bude preglednijje

In [96]:
total = len(valid_qa)

print(
    f"Training: {len(train_data)} "
    f"({len(train_data) / total:.2%})"
)

print(
    f"Validation: {len(validation_data)} "
    f"({len(validation_data) / total:.2%})"
)

print(
    f"Test: {len(test_data)} "
    f"({len(test_data) / total:.2%})"
)

Training: 100 (69.93%)
Validation: 21 (14.69%)
Test: 22 (15.38%)


In [97]:
train_ids = {
    example["id"]
    for example in train_data
}

validation_ids = {
    example["id"]
    for example in validation_data
}

test_ids = {
    example["id"]
    for example in test_data
}

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

assert len(
    train_ids | validation_ids | test_ids
) == len(valid_qa)

print("Nema preklapanja između skupova.")

Nema preklapanja između skupova.


In [98]:
#Čuvanje

In [99]:
PROCESSED_CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)

SPLITS_DIR = (
    PROJECT_ROOT
    / "data"
    / "splits"
)

SPLITS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [100]:
def save_jsonl(records: list[dict], path: Path) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )

In [103]:
save_jsonl(
    processed_chunks,
    PROCESSED_CHUNKS_PATH
)

TRAIN_PATH = SPLITS_DIR / "train.jsonl"
VALIDATION_PATH = SPLITS_DIR / "validation.jsonl"
TEST_PATH = SPLITS_DIR / "test.jsonl"

save_jsonl(
    train_data,
    TRAIN_PATH
)
save_jsonl(
    validation_data,
    VALIDATION_PATH
)
save_jsonl(
    test_data,
    TEST_PATH
)

Training skup: /home/jelena/Desktop/faks/masinsko/Student-Question-Answering-from-Course-Materials/data/splits/train.jsonl
Validation skup: /home/jelena/Desktop/faks/masinsko/Student-Question-Answering-from-Course-Materials/data/splits/validation.jsonl
Test skup: /home/jelena/Desktop/faks/masinsko/Student-Question-Answering-from-Course-Materials/data/splits/test.jsonl
